# Ollama + OLMX LLM Benchmark
This notebook automatically discovers local Ollama models by running `ollama list`. Optional OLMX models can be added in `OLMX_MODELS`.

1. Save your full CANBK request JSON as `benchmark_request.json` in the notebook's working directory.
2. Ensure `ollama list` works in the same environment.
3. Optionally add OLMX model IDs in the configuration cell.
4. Run the cells in order. Models run sequentially.

In [11]:
from pathlib import Path
import csv, json, os, subprocess, time, urllib.request, urllib.error
from datetime import datetime, timezone

REQUEST_FILE = Path('benchmark_request.json')
OUTPUT_DIR = Path('benchmark_results')
OLLAMA_URL = 'http://127.0.0.1:11434/api/chat'
OLMX_URL = 'http://127.0.0.1:8008/v1/chat/completions'
OLMX_API_KEY = "sk-omlx-cBRMujvOcaBgtMw7z9mmvGLi"  # set in your shell; do not hard-code
TEMPERATURE = 0
NUM_PREDICT = 8192
MAX_TOKENS = 8192
REQUEST_TIMEOUT_SECONDS = 1800
REQUEST_JSON_MODE = True

# Optional OLMX models; Ollama models are discovered automatically.
OLMX_MODELS = [
    {'name': 'OLMX model', 'model_id': 'gemma-4-12B-it-oQ4e-mtp'},
]

In [ ]:
def discover_ollama_models():
    try:
        result = subprocess.run(['ollama', 'list'], check=True, capture_output=True,
                                text=True, timeout=30)
    except FileNotFoundError:
        print('WARNING: ollama command not found'); return []
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as e:
        print('WARNING: ollama list failed:', e); return []
    lines = [line.strip() for line in result.stdout.splitlines() if line.strip()]
    models = []
    for line in lines[1:]:  # skip NAME / ID / SIZE / MODIFIED header
        fields = line.split()
        if fields:
            model_id = fields[0]
            models.append({'name': model_id, 'runtime': 'ollama', 'model_id': model_id})
    return models

def get_model_configs():
    models = discover_ollama_models()
    for item in OLMX_MODELS:
        mid = item.get('model_id', '').strip()
        if mid and 'YOUR_' not in mid:
            models.append({'name': item.get('name', mid), 'runtime': 'olmx', 'model_id': mid})
    seen, unique = set(), []
    for item in models:
        key = (item['runtime'], item['model_id'])
        if key not in seen: unique.append(item); seen.add(key)
    return unique

def post_json(url, payload, extra_headers=None):
    headers = {'Content-Type': 'application/json'}
    if extra_headers: headers.update(extra_headers)
    req = urllib.request.Request(url, data=json.dumps(payload).encode(),
          headers=headers, method='POST')
    with urllib.request.urlopen(req, timeout=REQUEST_TIMEOUT_SECONDS) as response:
        return json.loads(response.read().decode())

def call_model(config, messages):
    runtime, mid = config['runtime'], config['model_id']
    if runtime == 'ollama':
        p = {'model': mid, 'messages': messages, 'stream': False, 'think': False,
             'options': {'temperature': TEMPERATURE, 'num_predict': NUM_PREDICT}}
        if REQUEST_JSON_MODE: p['format'] = 'json'
        return post_json(OLLAMA_URL, p)
    p = {'model': mid, 'messages': messages, 'stream': False,
         'temperature': TEMPERATURE, 'max_tokens': MAX_TOKENS}
    if REQUEST_JSON_MODE: p['response_format'] = {'type': 'json_object'}
    headers = {'Authorization': f'Bearer {OLMX_API_KEY}'} if OLMX_API_KEY else {}
    return post_json(OLMX_URL, p, extra_headers=headers)

def extract_content(runtime, result):
    if runtime == 'ollama':
        content = (result.get('message') or {}).get('content', '')
        meta = {'prompt_tokens': result.get('prompt_eval_count'),
                'completion_tokens': result.get('eval_count'),
                'server_total_duration_ns': result.get('total_duration'),
                'finish_reason': result.get('done_reason')}
    else:
        choices = result.get('choices') or []; choice = choices[0] if choices else {}
        content = (choice.get('message') or {}).get('content', '')
        usage = result.get('usage') or {}
        meta = {'prompt_tokens': usage.get('prompt_tokens'),
                'completion_tokens': usage.get('completion_tokens'),
                'server_total_duration_ns': None, 'finish_reason': choice.get('finish_reason')}
    if isinstance(content, list): content = ''.join(p.get('text', '') if isinstance(p, dict) else str(p) for p in content)
    if not isinstance(content, str): content = json.dumps(content, ensure_ascii=False)
    return content, meta

def safe_name(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in s).strip('._') or 'model'

In [13]:
if not REQUEST_FILE.exists():
    raise FileNotFoundError('Save full CANBK request JSON as benchmark_request.json in this working directory.')
template = json.loads(REQUEST_FILE.read_text(encoding='utf-8'))
messages = template.get('messages')
if not isinstance(messages, list) or not messages:
    raise ValueError("benchmark_request.json must contain a non-empty 'messages' list")
models = get_model_configs()
if not models: raise RuntimeError('No models found; check `ollama list` or configure OLMX_MODELS.')
print('Sequential model queue:')
for m in models: print(f"  [{m['runtime']}] {m['name']}")
OUTPUT_DIR.mkdir(exist_ok=True)
summary = []
for i, config in enumerate(models, 1):
    name, runtime, mid = config['name'], config['runtime'], config['model_id']
    print(f'\\n[{i}/{len(models)}] Running {name} ({runtime})')
    started = datetime.now(timezone.utc).isoformat(); t0 = time.perf_counter()
    try:
        result = call_model(config, messages); elapsed = time.perf_counter() - t0
        content, meta = extract_content(runtime, result)
        try: parsed, valid, parse_error = json.loads(content), True, None
        except json.JSONDecodeError as e: parsed, valid, parse_error = None, False, f'{e.msg} at {e.pos}'
        path = OUTPUT_DIR / f'{safe_name(name)}_{runtime}.json'
        record = {'name': name, 'runtime': runtime, 'model_id': mid, 'started_at_utc': started,
                  'elapsed_seconds': round(elapsed, 3), 'valid_json': valid,
                  'json_parse_error': parse_error, **meta,
                  'response': parsed if valid else content, 'raw_api_response': result}
        path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding='utf-8')
        n = meta.get('completion_tokens')
        summary.append({'name': name, 'runtime': runtime, 'model_id': mid,
          'elapsed_seconds': round(elapsed, 3), 'prompt_tokens': meta.get('prompt_tokens'),
          'completion_tokens': n, 'approx_output_tokens_per_second': round(n/elapsed, 3) if n and elapsed else None,
          'valid_json': valid, 'finish_reason': meta.get('finish_reason'), 'error': None, 'result_file': str(path)})
        print(f'  {elapsed:.2f}s | valid JSON={valid} | output tokens={n}')
        if parse_error: print('  JSON parse issue:', parse_error)
    except Exception as e:
        elapsed = time.perf_counter() - t0; err = f'{type(e).__name__}: {e}'
        print('  FAILED:', err)
        summary.append({'name': name, 'runtime': runtime, 'model_id': mid, 'elapsed_seconds': round(elapsed, 3),
          'prompt_tokens': None, 'completion_tokens': None, 'approx_output_tokens_per_second': None,
          'valid_json': False, 'finish_reason': None, 'error': err, 'result_file': None})
if summary:
    (OUTPUT_DIR / 'results_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    with (OUTPUT_DIR / 'results_summary.csv').open('w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=list(summary[0].keys())); w.writeheader(); w.writerows(summary)
print('Finished. Results:', OUTPUT_DIR.resolve())

Sequential model queue:
  [ollama] hf.co/bartowski/Qwen_Qwen3-30B-A3B-GGUF:Q2_K
  [ollama] hf.co/abenzerps/Nemotron-3.5-Bolt-E4B-GSQ-RCO-GGUF:Q5_K_M
  [ollama] hf.co/nvidia/NVIDIA-Nemotron-3-Nano-4B-GGUF:Q4_K_M
  [ollama] hf.co/nvidia/NVIDIA-Nemotron-3.5-Bolt-E4B-4GB:Q4_K_M
  [ollama] qwen3-4b-instruct-16k:latest
  [olmx] OLMX model
\n[1/6] Running hf.co/bartowski/Qwen_Qwen3-30B-A3B-GGUF:Q2_K (ollama)
  18.49s | valid JSON=True | output tokens=614
\n[2/6] Running hf.co/abenzerps/Nemotron-3.5-Bolt-E4B-GSQ-RCO-GGUF:Q5_K_M (ollama)
  71.94s | valid JSON=True | output tokens=1950
\n[3/6] Running hf.co/nvidia/NVIDIA-Nemotron-3-Nano-4B-GGUF:Q4_K_M (ollama)
  295.25s | valid JSON=False | output tokens=8192
  JSON parse issue: Unterminated string starting at at 1445
\n[4/6] Running hf.co/nvidia/NVIDIA-Nemotron-3.5-Bolt-E4B-4GB:Q4_K_M (ollama)
  75.48s | valid JSON=True | output tokens=1876
\n[5/6] Running qwen3-4b-instruct-16k:latest (ollama)
  52.35s | valid JSON=True | output tokens=1370
\n[